In [ ]:
import pandas as pd
import plotly.express as px
from datetime import date, timedelta, datetime

# Read the CSV file into a DataFrame
df = pd.read_csv(
    "C:/Users/User/Documents/Github/Changed Appliance Project/occtopi-changed-appliance/data/raw/water-dispenser-full-data.csv")
df.columns = ['Timestamp', 'Value']
# Convert the 'Timestamp' column to datetime objects. The format is already a string representation of a datetime.
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.sort_values(by='Timestamp').reset_index(drop=True)

print("--- Calculating Daily Statistics ---")
# Group by the date part of 'Timestamp' and get descriptive statistics for 'Value'

daily_summary_stats_df = df.groupby(df['Timestamp'].dt.date)[
    'Value'].describe()

daily_summary_stats_df.index = pd.to_datetime(daily_summary_stats_df.index)
daily_summary_stats_df.index.name = 'Date'

print("\n--- Daily Statistics Summary DataFrame (first 5 rows) ---")
print(daily_summary_stats_df.head())

print("\n--- Plotting Daily Statistics Over Time ---")

fig_daily_stats = px.line(
    daily_summary_stats_df,
    title='Daily Power Usage Statistics Over Time',
    labels={'value': 'Statistic Value', 'variable': 'Statistic Metric'},
    markers=True
)
fig_daily_stats.show()


print("\n--- Calculating 4-Hourly Statistics ---")

df['4Hour_Interval'] = df['Timestamp'].dt.floor(
    '4h')  # Create a new column for 4-hour intervals

# Group by the 4-hour interval and get descriptive statistics
four_hourly_summary_stats_df = df.groupby('4Hour_Interval')['Value'].describe()

print("\n--- 4-Hourly Statistics Summary DataFrame (first 5 rows) ---")
print(four_hourly_summary_stats_df.head())

print("\n--- Plotting 4-Hourly Statistics Over Time ---")

fig_four_hourly_stats = px.line(
    four_hourly_summary_stats_df,
    title='4-Hourly Power Usage Statistics Over Time',
    labels={'value': 'Statistic Value', 'variable': 'Statistic Metric'},
    markers=True
)
fig_four_hourly_stats.show()

# Plot a normal chart of the dataframe
print("\n--- Overall Power Usage Chart (All Data) ---")
fig = px.line(df,
              x='Timestamp',
              y='Value',
              title='Power Usage Chart (All Data)',
              labels={'Timestamp': 'Time (hours) and Day', 'Value': 'Watts'},
              markers=False
              )
fig.show()


In [ ]:
import pandas as pd
import plotly.express as px
from datetime import date, timedelta, datetime

def load_and_process_4hourly_stats(file_path, file_label, stats_to_keep=['count', 'mean', 'std', 'min', 'max']):
    """
    Loads data from a CSV file, calculates specified 4-hourly statistics,
    and returns a long-format DataFrame.
    """
    # Read CSV, handling potential BOM (like in printer-data.csv)
    data_df = pd.read_csv(file_path, encoding='utf-8-sig')

    # Standardize column names: first column to 'Timestamp', second to 'Value'
    rename_map = {data_df.columns[0]: 'Timestamp', data_df.columns[1]: 'Value'}
    data_df.rename(columns=rename_map, inplace=True)

    # Convert 'Timestamp' to datetime objects
    data_df['Timestamp'] = pd.to_datetime(data_df['Timestamp'])
    # Ensure 'Value' is numeric, coercing errors and dropping NaNs that result
    data_df['Value'] = pd.to_numeric(data_df['Value'], errors='coerce')
    data_df.dropna(subset=['Value'], inplace=True)

    # Sort by 'Timestamp'
    data_df = data_df.sort_values(by='Timestamp').reset_index(drop=True)

    # Create a new column for 4-hour intervals
    data_df['4Hour_Interval'] = data_df['Timestamp'].dt.floor('4h')

    # Group by the 4-hour interval and get descriptive statistics
    summary_stats = data_df.groupby('4Hour_Interval')['Value'].describe()

    # Select only the required statistics and melt to long format
    summary_stats = summary_stats[stats_to_keep]
    summary_stats_long = summary_stats.reset_index().melt(
        id_vars=['4Hour_Interval'], var_name='Statistic_Metric', value_name='Statistic_Value')
    summary_stats_long['DataSource'] = file_label
    return summary_stats_long

# Read the CSV file into a DataFrame
df = pd.read_csv(
    "C:/Users/User/Documents/Github/Changed Appliance Project/occtopi-changed-appliance/data/raw/water-dispenser-full-data.csv")
df.columns = ['Timestamp', 'Value']
# Convert the 'Timestamp' column to datetime objects. The format is already a string representation of a datetime.
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.sort_values(by='Timestamp').reset_index(drop=True)


print("\n--- Calculating 4-Hourly Statistics ---")

df['4Hour_Interval'] = df['Timestamp'].dt.floor(
    '4h')  # Create a new column for 4-hour intervals

# Group by the 4-hour interval and get descriptive statistics
four_hourly_summary_stats_df = df.groupby('4Hour_Interval')['Value'].describe()

print("\n--- 4-Hourly Statistics Summary DataFrame (first 5 rows) ---")
print(four_hourly_summary_stats_df.head())

print("\n--- Plotting 4-Hourly Statistics Over Time ---")

fig_four_hourly_stats = px.line(
    four_hourly_summary_stats_df,
    title='4-Hourly Power Usage Statistics Over Time',
    labels={'value': 'Statistic Value', 'variable': 'Statistic Metric'},
    markers=True
)
fig_four_hourly_stats.show()

# Plot a normal chart of the dataframe
print("\n--- Overall Power Usage Chart (All Data) ---")
fig = px.line(df,
              x='Timestamp',
              y='Value',
              title='Power Usage Chart (All Data)',
              labels={'Timestamp': 'Time (hours) and Day', 'Value': 'Watts'},
              markers=False
              )
fig.show()

print("\n--- Comparing 4-Hourly Statistics for Water Dispenser and Printer ---")

# Define file paths
water_dispenser_file_path = "C:/Users/User/Documents/Github/Changed Appliance Project/occtopi-changed-appliance/data/raw/water-data.csv"
printer_file_path = "c:/Users/User/Documents/Github/Changed Appliance Project/occtopi-changed-appliance/data/raw/printer-data.csv" # Path from context

# Load and process stats for both datasets
stats_to_plot = ['count', 'mean', 'std', 'min', 'max'] # Excludes 25%, 50%, 75%

water_dispenser_stats_long = load_and_process_4hourly_stats(water_dispenser_file_path, "Water Dispenser", stats_to_keep=stats_to_plot)
printer_stats_long = load_and_process_4hourly_stats(printer_file_path, "Printer", stats_to_keep=stats_to_plot)

# Combine the statistics
combined_stats_df = pd.concat([water_dispenser_stats_long, printer_stats_long], ignore_index=True)

print("\n--- Combined Statistics DataFrame for Comparison (first 5 rows) ---")
print(combined_stats_df.head())

print("\n--- Plotting Comparison of 4-Hourly Statistics ---")

fig_comparison_stats = px.line(
    combined_stats_df,
    x='4Hour_Interval',
    y='Statistic_Value',
    color='DataSource',  # Differentiates lines by "Water Dispenser" or "Printer"
    facet_row='Statistic_Metric', # Creates subplots for each statistic
    title='Comparison of 4-Hourly Power Usage Statistics (Water Dispenser vs. Printer)',
    labels={'4Hour_Interval': 'Time Interval (4-hourly)', 'Statistic_Value': 'Value', 'DataSource': 'Appliance'},
    markers=True
)
fig_comparison_stats.update_yaxes(matches=None, title_text="") # Allow independent y-axes scales for each facet and remove individual y-axis titles for facets
fig_comparison_stats.update_layout(height=250 * len(stats_to_plot)) # Adjust height based on number of facets
fig_comparison_stats.show()


NameError: name 'plt' is not defined